## ChatGPT

Ask VQA to ChatGPT.

Docs:

* how to upload images: https://platform.openai.com/docs/guides/vision
* types of models: https://stackoverflow.com/questions/75774873/openai-api-error-this-is-a-chat-model-and-not-supported-in-the-v1-completions
* token limits: https://platform.openai.com/settings/organization/limits
* models: https://platform.openai.com/docs/models/model-endpoint-compatibility

In [1]:
#!pip install openai

In [2]:
# dir_api = '/Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/chatgpt_api/' #store API results for example data
# #model = "gpt-5-nano-2025-08-07"
# model = "gpt-5.4-nano-2026-03-17"
# temperature = 1.0 ## only allowed
# verbosity = None # default
# reasoning_level = None # default


# dir_api = '/Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/chatgpt_api_highR_lowVerbose/' #store API results for example data
# model = "gpt-5.4-nano-2026-03-17"
# temperature = 1.0 ## only allowed
# verbosity = 'low' # default
# reasoning_level = 'high' # default

dir_api = '/Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/chatgpt_api_highR_lowVerbose_mini/' #store API results for example data
model = "gpt-5.4-mini-2026-03-17"
temperature = 1.0 ## only allowed
verbosity = 'low' # default
reasoning_level = 'high' # default

key_file = '/Users/jnaiman/.openai/key.txt'

jsons_dir = '/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/qa_jsons/' # directory where jsons created with figure are stored
imgs_dir = '/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/imgs/' # where images are stored

# for saving temp images for reading in
tmp_dir = '/Users/jnaiman/Downloads/tmp/'

img_format = 'jpeg'

# for asking for reasoning
reasoning_text = 'In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, no other text.'

In [3]:
import openai
import base64
from openai import OpenAI
from PIL import Image
import numpy as np
import json
import re
import pickle
import os
from glob import glob

# debug
from importlib import reload

from sys import path
path.append('../')
import utils.llm_utils
reload(utils.llm_utils)
from utils.llm_utils import parse_qa, load_image, get_img_json_pair, parse_for_errors

from utils.plot_qa_utils import get_nplots
import time


In [4]:

def send_to_chatgpt(question_list, client, image_path, encoded_image,
                    model ="gpt-4o-mini", 
                    tmp_dir = '/Users/jnaiman/Downloads/tmp/',
                    test_run = True, fac=1.0, img_format='png',
                    verbose=True, temperature=1.0,
                    subset_questions_by_keys = None, reasoning = None, 
                    reasoning_level=None, verbosity=None,
                    max_retries=7):

    iFac = 2.0
    success = False
    is_subset = False
    new_fac = fac
    retry_count = 0

    while not success:
        try:
            question = question_list['context'] + " " + question_list['question'] + " " + question_list['format']
            if reasoning is not None:
                question += " " + reasoning
            question = question.lstrip()
            question = question[0].lower() + question[1:]
            if verbose: print('   on question:',question)
            prompt = f"I am going to show you an image. Now, {question}"
            prompt_save = f"I am going to show you an image. Now, {question}"

            if subset_questions_by_keys is not None and type(subset_questions_by_keys) == type([]):
                for s in subset_questions_by_keys:
                    if s in question:
                        is_subset = True
            else:
                is_subset = True

            messages = [
                {"role": "developer", "content": question_list['persona']},
                {"role": "user", "content": [
                    {
                        "type": "input_text",
                        "text": prompt
                    },
                    {
                        "type": "input_image",
                        "image_url": f"data:image/{img_format};base64,{encoded_image}"
                    }
                ]}
            ]

            if not test_run and is_subset:
                kwargs = {}
                if reasoning_level is not None:
                    kwargs["reasoning"] = {"effort": reasoning_level}
                if verbosity is not None:
                    kwargs["text"] = {"verbosity": verbosity}

                response = client.responses.create(
                    model=model,
                    input=messages,
                    **kwargs
                )
                success = True
            elif not is_subset:
                response = 'Not asked'
                success = True
            else:
                success = True

        except Exception as e:
            print('[ERROR]', e)
            error_str = str(e) # for later parsing

            if '429' in error_str:
                # rate limit — exponential backoff
                if retry_count >= max_retries:
                    print(f'Max retries ({max_retries}) reached on rate limit, giving up.')
                    raise
                wait_time = (2 ** retry_count) + 1  # 2, 3, 5, 9, 17 seconds
                print(f'Rate limit hit, waiting {wait_time}s before retry {retry_count+1}/{max_retries}...')
                time.sleep(wait_time)
                retry_count += 1
                # don't change the image size, just retry
            else:
                # other error (e.g. image too large) — resize and retry
                retry_count = 0  # reset backoff counter for non-rate-limit errors
                new_fac = fac/iFac
                print('new fac = ', new_fac)
                encoded_image = load_image(image_path, fac=new_fac, tmp_dir=tmp_dir, img_format=img_format)
                iFac += 1

    if not test_run and is_subset:
        answer = ""
        for item in response.output:
            if hasattr(item, "content") and item.content is not None:
                for block in item.content:
                    if hasattr(block, "text"):
                        answer += block.text

        question_list['raw answer'] = answer
        answer_format = answer.replace("```json\n",'').replace("\n```",'')
        try:
            question_list['Response'] = json.loads(answer_format)
        except:
            question_list['Response'] = answer_format
            question_list['Error'] = 'JSON formatting'
        question_list['Response String'] = answer_format
        success = True
    elif not is_subset:
        answer = response
        question_list['raw answer'] = answer
        question_list['Response'] = answer
        question_list['Response String'] = answer
        success = True
    else:
        question_list['Response'] = 'TEST RUN'
        question_list['Response String'] = 'TEST RUN'
    question_list['fac'] = new_fac

    return question_list, prompt_save

In [5]:
def print_qa(pickle_file, qa_in, subset_questions_by_keys=None, showNotAsked=False):
    if subset_questions_by_keys is not None:
        print('---------- ASKED ----------')
    print(pickle_file)
    print('*********')
    for qa_pairs in qa_in:
        hasSub = False
        if subset_questions_by_keys is not None and type(subset_questions_by_keys) == type([]):
            for s in subset_questions_by_keys:
                if s in qa_pairs['prompt']:
                    hasSub = True
        else:
            hasSub = True

        if hasSub:
            print('Prompt:', qa_pairs['prompt'])
            print('   Real A:', qa_pairs['A'])
            print('ChatGPT A:', qa_pairs['Response'])
            print('')

    if subset_questions_by_keys is not None and showNotAsked:
        print('')
        print('')
        print('------------ NOT ASKED -----------')
        for qa_pairs in qa_in:
            hasSub = False
            if subset_questions_by_keys is not None and type(subset_questions_by_keys) == type([]):
                for s in subset_questions_by_keys:
                    if s in qa_pairs['prompt']:
                        hasSub = True

            if not hasSub:
                print('Prompt:', qa_pairs['prompt'])
                print('   Real A:', qa_pairs['A'])
                print('ChatGPT A:', qa_pairs['Response'])
                print('')

In [6]:
# setup
with open(key_file,'r') as f:
    api_key = f.read()

client = OpenAI(
  api_key=api_key.strip(),  # this is also the default, it can be omitted
)

In [7]:
jsons_to_parse1 = glob(jsons_dir + '/*.json')
# clean
jsons_to_parse = []
for j in jsons_to_parse1:
    img_path = imgs_dir + j.split('/')[-1].removesuffix('_qa.json') + '.' + img_format
    if os.path.exists(img_path):
        jsons_to_parse.append(j)

print('total qa files:', len(jsons_to_parse))
jsons_to_parse[:3]

total qa files: 200


['/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/qa_jsons/Picture_000015_qa.json',
 '/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/qa_jsons/Picture_000005_qa.json',
 '/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/qa_jsons/Picture_000077_qa.json']

Look at a possible questions:

In [8]:
iping = 0

while iping < len(jsons_to_parse):
    try:
        json_path = jsons_to_parse[iping]
        iping = len(jsons_to_parse)
    except:
        iping += 1
        print(iping)

if iping < len(jsons_to_parse):
    verbose = False
    restart = False

    fac = 1.0

    img_path = imgs_dir + json_path.split('/')[-1].removesuffix('_qa.json') + '.' + img_format
    encoded_image, img_format_media, base_json, err = get_img_json_pair(img_path, json_path, dir_api, 
                                                        fac=fac, restart=restart, img_format=img_format,
                                                        tmp_dir=tmp_dir)
    print('Image format media:', img_format_media)
else:
    base_json = {'VQA':'NOT FOUND'}

base_json['VQA']

'NOT FOUND'

In [ ]:
iMax = 211 # should be number of jsons
verbose = False
test_run = False # run w/o actually pinging openai
restart = False

subset_questions_by_keys = None # ['median', 'ngaussians'] # set to None to do all questions

reload(utils.llm_utils)
from utils.llm_utils import parse_qa

fac = 1.0
for ijson,json_path in enumerate(jsons_to_parse):
    if ijson >= iMax:
        continue

    print('on', ijson+1, 'of', min(iMax,len(jsons_to_parse)))

    # get image and base json
    img_path = imgs_dir + json_path.split('/')[-1].removesuffix('_qa.json') + '.' + img_format
    print('Processing image:', img_path)
    encoded_image, img_format_media, base_json, err = get_img_json_pair(img_path, json_path, dir_api, 
                                                      fac=fac, restart=restart,img_format=img_format,
                                                      tmp_dir=tmp_dir)

    if err:
        continue


    ###### create QA ########
    qa = []
    
    # figure-level
    for k,v in base_json['VQA']['Level 1']['Figure-level questions'].items():
        out = {'Q':v['Q'], 'A':v['A'], 'Level':'Level 1', 'type':'Figure-level questions', 'Response':"", 
               "persona":v['persona'], 'context':v['context'], 'question':v['question'], 'format':v['format'],
               "reasoning":reasoning_text}
        qa.append(out)
    for level in ['Level 2', 'Level 3']:
        if level in base_json['VQA']:
            if 'Figure-level questions' in base_json['VQA'][level]:
                #print('** yes, level ***', level)
                for k,v in base_json['VQA'][level]['Figure-level questions'].items():
                    out = {'Q':v['Q'], 'A':v['A'], 'Level':level, 'type':'Figure-level questions', 'Response':"", 
                        "persona":v['persona'], 'context':v['context'], 'question':v['question'], 'format':v['format'],
                        "reasoning":reasoning_text}
                    qa.append(out)
                #import sys; sys.exit()
    # what kinds?
    #types = ['(words + list)', '(words)']
    types = []
    
    # get uniques
    level_parse = 'Level 1'
    plot_level = 'Plot-level questions'
    qa = parse_qa(level_parse, plot_level, qa, base_json['VQA'], types, use_split_keys=False)
    
    level_parse = 'Level 2'
    plot_level = 'Plot-level questions'
    qa = parse_qa(level_parse, plot_level, qa, base_json['VQA'], types, use_split_keys=False)
    
    level_parse = 'Level 3'
    plot_level = 'Plot-level questions'
    qa = parse_qa(level_parse, plot_level, qa, base_json['VQA'], types, use_split_keys=False)

    responses = []
    for question_list in qa:
        response, prompt = send_to_chatgpt(question_list, client, img_path, encoded_image,
                    model = model, img_format = img_format_media, temperature=temperature,
                    test_run = test_run, subset_questions_by_keys=subset_questions_by_keys, 
                    fac=fac, reasoning = reasoning_text, reasoning_level=reasoning_level, verbosity=verbosity)
        responses.append(response)
        question_list['prompt'] = prompt

    # parse for errors
    qa = parse_for_errors(qa)

    # dump to file
    if not test_run:
        with open(dir_api + json_path.split('/')[-1].removesuffix('.json')+ '.pickle', 'wb') as ff:
            pickle.dump([qa, model], ff)
        print("just saved:", dir_api + json_path.split('/')[-1].removesuffix('.json')+ '.pickle')
    else:
        print('Would store at:', dir_api + json_path.split('/')[-1].removesuffix('.json')+ '.pickle')

print("!!!!!! DONE !!!!!!!")


on 1 of 200
Processing image: /Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/imgs/Picture_000015.jpeg
   on question: how many panels are in this figure? Please format the output as a json as {"nrows":"", "ncols":""} to store the number of rows and columns. In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, no other text.
   on question: assume this is a figure made with matplotlib in Python.  Examples of plotting styles are "classic" or "ggplot". Examples of plotting styles are "classic" or "ggplot". What is the plot style used in this figure? Please format the output as a json as {"plot style":""} to store the matplotlib plotting style used in the figure. In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet f

In [ ]:
#(3.89-3.56)*200/10

## Look at data

Check out one, if you wanna:

In [ ]:
pickles = glob(dir_api + '*_qa.pickle')
pickles[:5]

['/Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/chatgpt_api_highR_lowVerbose/Picture_000005_qa.pickle',
 '/Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/chatgpt_api_highR_lowVerbose/Picture_000083_qa.pickle',
 '/Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/chatgpt_api_highR_lowVerbose/Picture_000067_qa.pickle',
 '/Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/chatgpt_api_highR_lowVerbose/Picture_000142_qa.pickle',
 '/Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/chatgpt_api_highR_lowVerbose/Picture_000199_qa.pickle']

In [ ]:
ifile = 5
with open(pickles[ifile], 'rb') as f:
    qa_in = pickle.load(f)[0]

In [ ]:
print_qa(pickles[ifile], qa_in, subset_questions_by_keys=subset_questions_by_keys, showNotAsked=False)

/Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/chatgpt_api_highR_lowVerbose/Picture_000130_qa.pickle
*********
Prompt: I am going to show you an image. Now, how many panels are in this figure? Please format the output as a json as {"nrows":"", "ncols":""} to store the number of rows and columns. In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, no other text.
   Real A: {'nrows': 1, 'ncols': 1}
ChatGPT A: {"nrows":"1","ncols":"1"}
{"explanation":"The image shows a single plotting area with one set of x/y axes and no visible grid of subplots or separate panels separated by borders."}

Prompt: I am going to show you an image. Now, assume this is a figure made with matplotlib in Python.  Examples of plotting styles are "classic" or "ggplot". Examples of plotting styles are "

In [ ]:
#qa_in[0]

{'Q': 'You are a helpful assistant that can analyze images.  How many panels are in this figure? Please format the output as a json as {"nrows":"", "ncols":""} to store the number of rows and columns.',
 'A': {'nrows': 1, 'ncols': 1},
 'Level': 'Level 1',
 'type': 'Figure-level questions',
 'Response': '{"nrows":"1","ncols":"1"}\n\n{"explanation":"The figure shows a single plotting area with one set of axes (one panel). There are no visible subplots or separated panels arranged in rows/columns."}',
 'persona': 'You are a helpful assistant that can analyze images.',
 'context': '',
 'question': 'How many panels are in this figure?',
 'format': 'Please format the output as a json as {"nrows":"", "ncols":""} to store the number of rows and columns.',
 'reasoning': 'In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer J